# Study 892 — Corporate-Bond Ladder — the teardown

The excess-of-cash Sharpe race with block-bootstrap CIs, the Newey-West diff *t*, the era cut, the 2022 calendar-year stress row, the costed net, and the 20-seed synthetic control.

In [1]:
R = {'start': '2007-06-30', 'end': '2026-06-30', 'n_months': 229, 'fingerprint': '4eeca3d56739', 'ew_dur': 7.5, 'dm_dur': 6.0, 'agg_dur': 6.0, 'lad_ann': 3.02, 'lad_sharpe': 0.334, 'lad_ci': (-0.13, 0.81), 'lad_dd': -18.3, 'agg_ann': 3.07, 'agg_sharpe': 0.386, 'agg_ci': (-0.08, 0.9), 'agg_dd': -17.1, 'bnd_ann': 3.1, 'bnd_sharpe': 0.396, 'diff_ann': -0.01, 'diff_bps_mo': -0.11, 't_hac': -0.02, 't_1s': -0.02, 'diff_sharpe': -0.005, 'diff_sharpe_ci': (-0.48, 0.46), 'ew_ann': 2.92, 'ew_sharpe': 0.276, 'ew_dd': -23.2, 'era1': '2007-2015', 'era1_diff': 0.67, 'era1_t': 0.57, 'era2': '2016-2021', 'era2_diff': -0.6, 'era2_t': -0.87, 'era3': '2022-2026', 'era3_diff': -0.53, 'era3_t': -1.09, 'y2021_lad': -2.75, 'y2021_agg': -1.77, 'y2021_gap': -0.99, 'y2022_lad': -12.42, 'y2022_agg': -13.02, 'y2022_gap': 0.6, 'y2023_lad': 3.9, 'y2023_agg': 5.66, 'y2023_gap': -1.75, 'cost1': 0.9, 'net1': -0.02, 'tnet1': -0.04, 'cost2': 3.0, 'net2': -0.04, 'tnet2': -0.07, 'null_t_mean': 0.2, 'null_t_sd': 1.36, 'null_fire': 3, 'planted_recovered': 1.22, 'planted_t': 3.99}

## Headline — duration-matched ladder vs AGG (excess of BIL)

In [2]:
print(f"ladder (dur {R['dm_dur']}y): ann {R['lad_ann']:+.2f}%  exSharpe "
      f"{R['lad_sharpe']:.3f} CI {R['lad_ci']}  maxDD {R['lad_dd']:.1f}%")
print(f"AGG    (dur {R['agg_dur']}y): ann {R['agg_ann']:+.2f}%  exSharpe "
      f"{R['agg_sharpe']:.3f} CI {R['agg_ci']}  maxDD {R['agg_dd']:.1f}%")
print(f"BND    (dur 5.9y): ann {R['bnd_ann']:+.2f}%  exSharpe {R['bnd_sharpe']:.3f}")
print(f"ladder - fund : {R['diff_ann']:+.2f}%/yr ({R['diff_bps_mo']:+.2f} bps/mo)  "
      f"HAC t = {R['t_hac']:+.2f}  1-sample t = {R['t_1s']:+.2f}")
print(f"diff-Sharpe   : {R['diff_sharpe']:+.3f}  CI {R['diff_sharpe_ci']} (straddles 0)")

ladder (dur 6.0y): ann +3.02%  exSharpe 0.334 CI (-0.13, 0.81)  maxDD -18.3%
AGG    (dur 6.0y): ann +3.07%  exSharpe 0.386 CI (-0.08, 0.9)  maxDD -17.1%
BND    (dur 5.9y): ann +3.10%  exSharpe 0.396
ladder - fund : -0.01%/yr (-0.11 bps/mo)  HAC t = -0.02  1-sample t = -0.02
diff-Sharpe   : -0.005  CI (-0.48, 0.46) (straddles 0)


## The naive equal-weight ladder — longer duration, worse Sharpe

In [3]:
print(f"EW ladder (dur {R['ew_dur']}y): ann {R['ew_ann']:+.2f}%  "
      f"exSharpe {R['ew_sharpe']:.3f}  maxDD {R['ew_dd']:.1f}%  -> LOSES to AGG")

EW ladder (dur 7.5y): ann +2.92%  exSharpe 0.276  maxDD -23.2%  -> LOSES to AGG


## Era cut — a real premium is stable; this one flips sign, never clears |t|>=1.1

In [4]:
for e,d,t in [(R['era1'],R['era1_diff'],R['era1_t']),
              (R['era2'],R['era2_diff'],R['era2_t']),
              (R['era3'],R['era3_diff'],R['era3_t'])]:
    print(f"{e}: ladder - fund {d:+.2f}%/yr  (HAC t {t:+.2f})")

2007-2015: ladder - fund +0.67%/yr  (HAC t +0.57)
2016-2021: ladder - fund -0.60%/yr  (HAC t -0.87)
2022-2026: ladder - fund -0.53%/yr  (HAC t -1.09)


## 2022 rate shock — a one-year credit-composition dodge, reversed in 2023

In [5]:
for yr,lad,agg,gap in [(2021,R['y2021_lad'],R['y2021_agg'],R['y2021_gap']),
                       (2022,R['y2022_lad'],R['y2022_agg'],R['y2022_gap']),
                       (2023,R['y2023_lad'],R['y2023_agg'],R['y2023_gap'])]:
    print(f"{yr}: ladder {lad:+.2f}%  AGG {agg:+.2f}%  gap {gap:+.2f} pp")

2021: ladder -2.75%  AGG -1.77%  gap -0.99 pp
2022: ladder -12.42%  AGG -13.02%  gap +0.60 pp
2023: ladder +3.90%  AGG +5.66%  gap -1.75 pp


## Tradability — the ladder is rolled annually; the one-ticker fund is free

In [6]:
for c,n,t in [(R['cost1'],R['net1'],R['tnet1']),(R['cost2'],R['net2'],R['tnet2'])]:
    print(f"ladder cost {c:.1f} bps/yr -> net diff {n:+.2f}%/yr (HAC t {t:+.2f})")

ladder cost 0.9 bps/yr -> net diff -0.02%/yr (HAC t -0.04)
ladder cost 3.0 bps/yr -> net diff -0.04%/yr (HAC t -0.07)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted premium.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from bond_ladder import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_world(edge_annual=0.0, seed=892+s))['t_hac'] for s in range(8)])
print(f"null (edge=0), 8 seeds: HAC t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_world(edge_annual=0.015, seed=892))
print(f"planted (+1.5%%/yr): recovered {planted['diff_ann_pct']:+.2f}%%/yr, HAC t = {planted['t_hac']:+.2f}")

null (edge=0), 8 seeds: HAC t mean +0.06 (sd 1.19), |t|>=2 in 1/8
planted (+1.5%%/yr): recovered +1.22%%/yr, HAC t = +3.99


## Verdict

- **Signal — None.** Duration-matched, ladder − fund = **-0.01%/yr** (HAC *t* = **-0.02**); the difference-Sharpe CI **(-0.48, 0.46)** straddles zero; the sign flips across eras (+0.67 / -0.60 / -0.53 %/yr, all |*t*| < 1.1). The naive equal-weight ladder underperforms (Sharpe 0.276 vs 0.386) purely on extra duration. Held-to-maturity vs mark-to-market is an accounting identity for default-free bonds; the 20-seed synthetic control recovers a *planted* premium at *t* = +3.99, so the null result is genuine. Short ~19-year survivor tape.
- **Tradability — Mirage.** No gross edge, and the ETF ladder pays 0.9–3.0 bps/yr of roll cost the buy-and-hold fund does not, so net it trails (-0.02 to -0.04%/yr). The ladder buys **behavioral comfort**, not risk-adjusted return.